# 原始 Stage1 A_0 0bp 开盘 EXECUTE/HOLD 局部净价值归因

比较相同09:45实物状态下执行冻结原目标与HOLD，次日按同一原账户动作复位；不可视为完整策略反事实。

In [1]:
import os,json,hashlib,copy,tarfile,tempfile,subprocess,shlex
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from src.backtest.ibkr_tiered import TieredFees
from src.backtest.raw_corporate_policy import settle_raw
from src.workflow.intraday_recovery_direct_allocation import project_legal_weights
from src.data.nas_market_data import archive_nas_recorder,_settings
root=Path.cwd();card=root/'research/stage1_original_0bp_open_hold_local_value_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
review_path=card/'design_review_final.json';assert review_path.exists();review=json.loads(review_path.read_text());notebook=json.loads((card/(card.name+'.ipynb')).read_text());cell_hash=hashlib.sha256(''.join(notebook['cells'][1]['source']).encode()).hexdigest()
assert review['verdict']=='APPROVE_DESIGN' and review['plan_sha256']==sha(card/'plan.json') and review['code_cell_sha256']==cell_hash
assert plan['frozen'] and not (card/'run_state.json').exists() and not (card/'qlib_recorder').exists()
s=plan['source'];assert sha(root/s['A0_summary'])==s['A0_summary_sha256'];summary=json.loads((root/s['A0_summary']).read_text());assert summary['recorder_id']==s['A0_recorder_id'] and summary['nas_evidence']['sha256']==s['A0_NAS_archive_sha256']
rb=json.loads((root/s['A0_NAS_readback']).read_text());assert rb['status']=='passed' and rb['archive_sha256']==s['A0_NAS_archive_sha256'] and rb['all_artifact_hashes_match']
a=Path(s['A0_recorder_uri'])/s['A0_experiment_id']/s['A0_recorder_id']/'artifacts'
for name,digest in s['A0_artifact_sha256'].items():assert sha(a/name)==digest
for name,digest in s['module_sha256'].items():assert sha(root/name)==digest
assert sha(root/s['prior_observed_fee_parity'])==s['prior_observed_fee_parity_sha256'] and sha(root/s['prior_executed_projection_parity'])==s['prior_executed_projection_parity_sha256']
fee_prior=json.loads((root/s['prior_observed_fee_parity']).read_text());projection_prior=json.loads((root/s['prior_executed_projection_parity']).read_text());assert fee_prior['status']=='A0_0bp_observed_Tiered_fee_parity_passed' and fee_prior['observed_orders']==1275 and max(fee_prior['max_abs_diff_by_field'].values())==0
assert projection_prior['status']=='original_A0_executed_legal_projection_parity_passed' and projection_prior['executed_rows']==797 and projection_prior['HOLD_rows']==147 and projection_prior['max_abs_weight_error']==0
state={'status':'started','plan_sha256':sha(card/'plan.json'),'code_cell_sha256':cell_hash,'new_model_fits':0,'new_native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/stage1_domain_heads_20261018'),region='us',kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time=plan['matched_trade_dates'][0],end_time=plan['matched_trade_dates'][1],freq='day'));assert len(calendar)==plan['expected_sessions']
with R.uri_context(s['A0_recorder_uri']):
 rec=R.get_recorder(recorder_id=s['A0_recorder_id'],experiment_id=s['A0_experiment_id']);native=rec.load_object('native/A_active_research_replay_0');frame=rec.load_object('inputs/frame');parent_plan=rec.load_object('inputs/plan');recovery=rec.load_object('inputs/recovery_spec');schedule=rec.load_object('inputs/schedule')
auth=native['authority'].reset_index(drop=True);targets=native['targets'].reset_index(drop=True);points=native['points'];orders=native['orders'].reset_index(drop=True)
for x in (auth,targets,frame):x['trade_date']=pd.to_datetime(x.trade_date).dt.normalize()
assert len(auth)==len(targets)==len(frame)==944 and auth.trade_date.equals(frame.trade_date) and auth.trade_date.equals(targets.trade_date) and auth.trade_date.equals(pd.Series(calendar,name='trade_date'))
assert not auth.recovery_action.any() and int(auth.execute_open_target.sum())==797 and int((~auth.execute_open_target).sum())==147
point0945=points.loc[pd.to_datetime(points.stamp).dt.strftime('%H:%M').eq('09:45')].copy();point0945['trade_date']=pd.to_datetime(point0945.stamp).dt.normalize();point0945=point0945.set_index('trade_date')
point0947=points.loc[pd.to_datetime(points.stamp).dt.strftime('%H:%M').eq('09:47')].copy();point0947['trade_date']=pd.to_datetime(point0947.stamp).dt.normalize();point0947=point0947.set_index('trade_date')
assert point0945.loc[calendar].index.equals(calendar) and point0947.loc[calendar].index.equals(calendar)
fee_state=TieredFees(parent_plan['tariff'],slippage_bps=0);order_at=pd.to_datetime(orders.stamp);order_i=0;daily=[];max_known_nav_gap=0.;max_current_q_gap=0.;max_current_cash_gap=0.;max_current_nav_gap=0.;max_next_q_gap=0.;max_next_cash_gap=0.;max_next_nav_gap=0.;corp_unknown=0;terminal_unknown=0
for i in range(944):
 row=frame.iloc[i];day=pd.Timestamp(row.trade_date);stamp=day+pd.Timedelta(hours=9,minutes=47)
 while order_i<len(orders) and order_at.iloc[order_i]<stamp:
  o=orders.iloc[order_i];fee_state.commit(int(o.raw_shares),stamp=o.stamp,cap_applied=bool(o.cap_applied));order_i+=1
 before=point0945.loc[day];q=before[['q_soxl','q_soxx']].to_numpy(float);cash=float(before.cash);receivable=float(before.receivable);knownprice=row[['raw_known_soxl','raw_known_soxx']].to_numpy(float);price=row[['raw_soxl_open','raw_soxx_open']].to_numpy(float)
 known_nav=float(q@knownprice+cash+receivable);known_gap=abs(known_nav-float(before.nav));max_known_nav_gap=max(max_known_nav_gap,known_gap);assert known_gap<=0.01
 target=project_legal_weights(float(row.soxl_weight),float(row.soxx_weight),float(auth.iloc[i].current_beta_cap),minimum_cash_weight=float(recovery['minimum_cash_weight']),maximum_soxl_weight=float(recovery['maximum_soxl_weight']))
 branches=[]
 for desired in (target,None):
  branch_fee=copy.deepcopy(fee_state);entry_q,entry_cash,_=settle_raw(q,cash,receivable,desired=desired,prices=price,stamp=stamp,fees=branch_fee);entry_nav=float(entry_q@price+entry_cash+receivable);branches.append({'q':entry_q,'cash':entry_cash,'fee_state':branch_fee,'nav':entry_nav})
 chosen=branches[0] if bool(auth.iloc[i].execute_open_target) else branches[1];saved_current=point0947.loc[day]
 q_gap=float(np.max(np.abs(chosen['q']-saved_current[['q_soxl','q_soxx']].to_numpy(float))));cash_gap=abs(float(chosen['cash'])-float(saved_current.cash));nav_gap=abs(chosen['nav']-float(saved_current.nav))
 max_current_q_gap=max(max_current_q_gap,q_gap);max_current_cash_gap=max(max_current_cash_gap,cash_gap);max_current_nav_gap=max(max_current_nav_gap,nav_gap)
 assert q_gap<=1e-8 and cash_gap<=0.02 and nav_gap<=0.02, f'current original action parity failed {day}: {q_gap} {cash_gap} {nav_gap}'
 value_bp=np.nan;label_at=pd.NaT;status='unknown_no_next_Stage1_trade_date' if i+1==944 else 'unknown_next_date_corporate_action' if pd.Timestamp(frame.iloc[i+1].trade_date) in schedule else 'mature_local_label'
 if status=='unknown_no_next_Stage1_trade_date':terminal_unknown+=1
 elif status=='unknown_next_date_corporate_action':corp_unknown+=1
 else:
  nxt=frame.iloc[i+1];nextday=pd.Timestamp(nxt.trade_date);nextstamp=nextday+pd.Timedelta(hours=9,minutes=47);nextprice=nxt[['raw_soxl_open','raw_soxx_open']].to_numpy(float);next_target=targets.iloc[i+1][['soxl_weight','soxx_weight','cash_weight']].to_numpy(float) if bool(auth.iloc[i+1].execute_open_target) else None
  reset=[]
  for branch in branches:
   rq,rc,_=settle_raw(branch['q'],branch['cash'],receivable,desired=next_target,prices=nextprice,stamp=nextstamp,fees=branch['fee_state']);reset.append({'q':rq,'cash':rc,'nav':float(rq@nextprice+rc+receivable)})
  actual=reset[0] if bool(auth.iloc[i].execute_open_target) else reset[1];saved_next=point0947.loc[nextday]
  q_gap=float(np.max(np.abs(actual['q']-saved_next[['q_soxl','q_soxx']].to_numpy(float))));cash_gap=abs(actual['cash']-float(saved_next.cash));nav_gap=abs(actual['nav']-float(saved_next.nav))
  max_next_q_gap=max(max_next_q_gap,q_gap);max_next_cash_gap=max(max_next_cash_gap,cash_gap);max_next_nav_gap=max(max_next_nav_gap,nav_gap)
  assert q_gap<=1e-8 and cash_gap<=0.02 and nav_gap<=0.02, f'next original chosen branch parity failed {day}: {q_gap} {cash_gap} {nav_gap}'
  value_bp=(reset[0]['nav']-reset[1]['nav'])/known_nav*10000;assert np.isfinite(value_bp);label_at=nextstamp+pd.Timedelta(minutes=2)
 daily.append({'trade_date':day,'original_execute':bool(auth.iloc[i].execute_open_target),'status':status,'label_available_at':label_at,'execute_minus_HOLD_bp':value_bp,'known_nav':known_nav,'risk_state':int(auth.iloc[i].risk_state),'current_beta_cap':float(auth.iloc[i].current_beta_cap)})
result=pd.DataFrame(daily);assert len(result)==944 and result.trade_date.equals(pd.Series(calendar,name='trade_date')) and int(result.original_execute.sum())==797
labels=int(result.execute_minus_HOLD_bp.notna().sum());assert labels+corp_unknown+terminal_unknown==944 and labels>=850
result['year']=result.trade_date.dt.year;result['quarter']=result.trade_date.dt.to_period('Q').astype(str)
def summarize(g):
 h=g.loc[~g.original_execute,'execute_minus_HOLD_bp'];e=g.loc[g.original_execute,'execute_minus_HOLD_bp'];return {'sessions':len(g),'mature_labels':int(g.execute_minus_HOLD_bp.notna().sum()),'HOLD_days':int((~g.original_execute).sum()),'HOLD_mature_labels':int(h.notna().sum()),'HOLD_mean_execute_minus_HOLD_bp':float(h.mean()) if h.notna().any() else None,'HOLD_positive_execute_advantage_days':int((h>0).sum()),'EXECUTE_days':int(g.original_execute.sum()),'EXECUTE_mature_labels':int(e.notna().sum()),'EXECUTE_mean_execute_minus_HOLD_bp':float(e.mean()) if e.notna().any() else None,'EXECUTE_negative_execute_advantage_days':int((e<0).sum())}
annual=[{'year':int(y),'complete_year':y in (2024,2025),**summarize(g)} for y,g in result.groupby('year',sort=True)];quarters=[{'quarter':q,**summarize(g)} for q,g in result.groupby('quarter',sort=True)];rolling=[]
for start in range(len(result)-125):
 g=result.iloc[start:start+126];rolling.append({'start_trade':str(g.iloc[0].trade_date.date()),'end_trade':str(g.iloc[-1].trade_date.date()),**summarize(g)})
assert len(annual)==4 and len(quarters)==16 and len(rolling)==819
worst126=result.loc[result.trade_date.between(pd.Timestamp('2023-08-01'),pd.Timestamp('2024-01-30'))];assert len(worst126)==126
predefined={'worst126':summarize(worst126),'2024Q3':summarize(result.loc[result.quarter.eq('2024Q3')]),'2026Q3':summarize(result.loc[result.quarter.eq('2026Q3')])}
static=pd.concat({'label':result.set_index('trade_date')[['execute_minus_HOLD_bp']],'meta':result.set_index('trade_date')[['original_execute','label_available_at']]},axis=1);static.index=pd.MultiIndex.from_arrays([result.trade_date,['SOXL_SOXX_CASH']*len(result)],names=['datetime','instrument'])
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'held':tuple(plan['matched_trade_dates'])});prepared=dataset.prepare('held',col_set=['label','meta']);assert len(prepared)==944
summary={'experiment':card.name,'plan_sha256':sha(card/'plan.json'),'decision':'descriptive_original_Stage1_A0_0bp_open_HOLD_local_value_no_policy_change','objective_achieved':False,'matched_sessions':944,'mature_local_labels':labels,'unknown_next_corporate_action':corp_unknown,'unknown_no_next_date':terminal_unknown,'all_days':summarize(result),'annual':annual,'quarterly':quarters,'predefined_examples':predefined,'max_known_raw_nav_gap_usd':max_known_nav_gap,'max_current_actual_q_gap':max_current_q_gap,'max_current_actual_cash_gap_usd':max_current_cash_gap,'max_current_actual_nav_gap_usd':max_current_nav_gap,'max_next_actual_q_gap':max_next_q_gap,'max_next_actual_cash_gap_usd':max_next_cash_gap,'max_next_actual_nav_gap_usd':max_next_nav_gap,'new_model_fits':0,'new_native_accounts':0,'local_values_are_full_strategy_value':False,'historical_independence':False,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/daily','analysis/annual','analysis/quarters','analysis/rolling126']}}
with R.start(experiment_name=card.name,uri=str(card/'qlib_recorder')):
 recorder=R.get_recorder();state.update(recorder_id=recorder.id,experiment_id=str(recorder.experiment_id));(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
 dataset.config(dump_all=True,recursive=True);recorder.save_objects(artifact_path='inputs',plan=plan,dataset=dataset,fee_parity=fee_prior,projection_parity=projection_prior);recorder.save_objects(artifact_path='analysis',daily=result,annual=annual,quarters=quarters,rolling126=rolling)
 recorder.log_metrics(HOLD_mature_labels=summary['all_days']['HOLD_mature_labels'],HOLD_mean_execute_minus_HOLD_bp=summary['all_days']['HOLD_mean_execute_minus_HOLD_bp'],EXECUTE_mature_labels=summary['all_days']['EXECUTE_mature_labels'],new_model_fits=0,new_native_accounts=0)
 with R.uri_context(str(card/'qlib_recorder')):reread=R.get_recorder(recorder_id=recorder.id,experiment_id=str(recorder.experiment_id)).load_object('analysis/daily')
 assert reread.equals(result)
summary.update(recorder_id=state['recorder_id'],experiment_id=state['experiment_id'],recorder_uri=str(card/'qlib_recorder'));(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':card.name,'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n');evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=state['recorder_id'],experiment_id=state['experiment_id'])
with tempfile.TemporaryDirectory(prefix='stage1-open-hold-restore-') as td:
 archive=Path(td)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(evidence['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==evidence['sha256']
 with tarfile.open(archive,'r:gz') as t:
  names=[m.name for m in t.getmembers() if m.isfile()];assert any(x.endswith('/artifacts/analysis/daily') for x in names) and any(x.endswith('/artifacts/inputs/dataset') for x in names)
summary['nas_evidence']=evidence;summary['nas_readback']={'sha256':evidence['sha256'],'verified':True};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');state['status']='completed';state['NAS_archive_sha256']=evidence['sha256'];(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
print({'decision':summary['decision'],'mature_labels':labels,'all_days':summary['all_days'],'max_next_actual_nav_gap_usd':max_next_nav_gap,'recorder_id':state['recorder_id'],'NAS_sha256':evidence['sha256']})


2026/10/23 05:03:59 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[88871:MainThread](2026-10-23 05:04:00,779) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[88871:MainThread](2026-10-23 05:04:00,781) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[88871:MainThread](2026-10-23 05:04:00,781) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_domain_heads_20261018')}


...

[88871:MainThread](2026-10-23 05:04:02,196) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_original_0bp_open_hold_local_value_20261023.


[88871:MainThread](2026-10-23 05:04:02,198) INFO - qlib.workflow - [exp.py:258] - Experiment 212779304661421894 starts running ...


[88871:MainThread](2026-10-23 05:04:02,271) INFO - qlib.workflow - [recorder.py:345] - Recorder 0968f1d2a5cd4af38af720c6fe671bec starts running under Experiment 212779304661421894 ...


[88871:MainThread](2026-10-23 05:04:03,204) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


{'decision': 'descriptive_original_Stage1_A0_0bp_open_HOLD_local_value_no_policy_change', 'mature_labels': 889, 'all_days': {'sessions': 944, 'mature_labels': 889, 'HOLD_days': 147, 'HOLD_mature_labels': 139, 'HOLD_mean_execute_minus_HOLD_bp': 0.3129082569161164, 'HOLD_positive_execute_advantage_days': 67, 'EXECUTE_days': 797, 'EXECUTE_mature_labels': 750, 'EXECUTE_mean_execute_minus_HOLD_bp': -0.053795549340447675, 'EXECUTE_negative_execute_advantage_days': 341}, 'max_next_actual_nav_gap_usd': 0.0, 'recorder_id': '0968f1d2a5cd4af38af720c6fe671bec', 'NAS_sha256': '87b0aafba2396e152266138ac81f49bb629c0bb30020adad58d88eba5b9c75fd'}


# 原始 Stage1 A_0：0bp 开盘 EXECUTE/HOLD 局部净价值归因

结论：`descriptive_original_Stage1_A0_0bp_open_HOLD_local_value_no_policy_change`。冻结原 A_0 的 944 个交易日中，889 日可在同一 09:45 实物状态、0bp 额外滑点且照收 IBKR Tiered 费用下，比较 09:47 执行原合法目标与 HOLD，并于次日 09:47 按相同原策略动作复位。54 个次日公司行动日和最后 1 日原位保留未知；0 拟合、0 新账户、0 重放。

原账户 797 日 EXECUTE、147 日 HOLD。成熟的 139 个 HOLD 日里，若改为执行，平均局部净优势仅 +0.313bp，67 日为正；750 个成熟 EXECUTE 日中，执行相对 HOLD 平均 -0.054bp，341 日为负。两者的均值均很小，不能把局部值相加成闭环财富。

预定弱区间 2023-08-01 至 2024-01-31 出场的 126 日内，31 个成熟 HOLD 日若改执行，平均 -1.549bp，故 HOLD 本身不解释该段亏损；87 个成熟 EXECUTE 日平均 +2.877bp。2024Q3 的成熟 EXECUTE 平均 +0.316bp；2026Q3 则为 -9.410bp（57 日），提示这段的开盘动作局部不利，但事后改为 HOLD 不能直接当交易规则。

机械证据：旧 1,275 笔订单的 Tiered 费用字段逐项精确复算；原 797 个 EXECUTE 目标投影精确一致。当前原选分支在当日和次日 09:47 的实物股数、可支配现金、净值最大误差均为 0；09:45 原价净值也完全闭合。全 4 年、16 季、819 个连续 126 日窗口保存在 Recorder。

这项诊断把主要研究焦点继续指向原配置目标、风险投影及其 SOXL 相对选择，而不是把 147 个 HOLD 一概删掉。局部共同复位使用次日原账户目标，只是归因工具；它不计不同动作对后续 NAV、风险状态和模型决策的反馈，不授予 0bp SOXX 超额或 2/5bp/前向资格。

原生 Qlib Recorder `0968f1d2a5cd4af38af720c6fe671bec` 已目标读回；NAS 归档 SHA-256 `87b0aafba2396e152266138ac81f49bb629c0bb30020adad58d88eba5b9c75fd` 经限定恢复四件工件逐项哈希核验。下一步在原 Stage1 最差 126 日与 2024Q3/2026Q3 的目标→投影→执行链中寻找**决策前**能区分现金机会和 SOXL 相对选择的不同信息，冻结后才运行完整 0bp 账户。